## Csomagok betöltése 

In [1]:
import copy
import math
import random
import numpy as np
import torch
import torch.nn as nn
from tensorflow.python.framework.random_seed import set_seed
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.models as models
import medmnist
from medmnist import INFO
import torch.nn.functional as F
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix
import os
import json
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wasserstein_distance



## Random seed értékek beállítása

**Később használva lesznek a backbone és a single-task tanításnál**

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Használt eszköz: {device}")


Használt eszköz: cuda


## Train - Validation - Test adathalmazok szétválasztása a hivatalos benchmark szerint

In [3]:
def get_dataloaders(dataset_name, batch_size=128, max_train_samples=None, max_eval_samples=None):
    info = INFO[dataset_name]
    DataClass = getattr(medmnist, info['python_class'])

    data_transform = transforms.Compose([ # Normálja és skálázza az adatokat
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    # A hivatalos MNIST splitek vannak stratidy szempontból jók
    train_dataset = DataClass(split='train', transform=data_transform, download=True, as_rgb=True)
    val_dataset   = DataClass(split='val',   transform=data_transform, download=True, as_rgb=True)
    test_dataset  = DataClass(split='test',  transform=data_transform, download=True, as_rgb=True)

    # Shuffle keveri hogy ne tanuljon rá a mintára
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,drop_last=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,drop_last=True)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,drop_last=True)


    return train_loader, val_loader, test_loader, info

## TxT performance mátrix vizualizációja és mentése

A kísérlet során a modell szekvenciálisan, egymás után tanulja meg a kijelölt $T$ darab feladatot egy előre meghatározott sorrendben:
$$\mathcal{T} = \{\tau_1, \tau_2, \dots, \tau_T\}$$

Minden egyes $i$-edik feladat ($\tau_i$) betanításának befejezése után a modell azonnali kiértékelésre kerül az összes feladat ($\tau_j$, ahol $j \in \{1, \dots, T\}$) független tesztkészletén.

A kiértékelés eredményét az $R \in \mathbb{R}^{T \times T}$ mátrix rögzíti, ahol:
* **Sorindex ($i \in \{1, \dots, T\}$):** A tanítási fázis, azaz az utoljára betanított feladat sorszáma.
* **Oszlopindex ($j \in \{1, \dots, T\}$):** A tesztelt feladat azonosítója.
* ***$R_{i,j}$:*** A modell tesztpontossága a $\tau_j$ feladaton, közvetlenül a $\tau_i$ feladat tanulásának lezárultakor.

$$R = \begin{bmatrix}
R_{1,1} & R_{1,2} & \dots & R_{1,T} \\
R_{2,1} & R_{2,2} & \dots & R_{2,T} \\
\vdots  & \vdots  & \ddots & \vdots  \\
R_{T,1} & R_{T,2} & \dots & R_{T,T}
\end{bmatrix}$$

* **Főátló ($R_{j,j}$):** Az egyes feladatok önálló betanulásakor elért **azonnali pontossága**.

* **Főátló alatti elemek ($i > j$):** A korábban megtanult feladatok ($\tau_j$) későbbi teljesítménye az újabb feladatok ($\tau_i$) elsajátítása után. Az oszlopok lefelé történő mozgása írja le a **catastrophic forgetting** dinamikáját.

* **Főátló feletti elemek ($i < j$):** A modell **előzetes teljesítménye** olyan feladatokon, amelyeket a tanítási folyamatban formálisan még nem látott.

* **Utolsó sor ($R_{T,j}$):** A teljes tanítási folyamat végén megmaradt végső modell tudásállapota az összes feladaton.

In [4]:
RESULT_DIR = os.path.abspath("results")
os.makedirs(RESULT_DIR, exist_ok=True)


def save_publication_matrix(
    R_matrix, task_order, experiment_name="sequential_baseline"
):
    plt.close("all")
    plt.rcParams.update(plt.rcParamsDefault)
    sns.set_theme(style="white")

    row_labels = [f"Step {i+1} ({t})" for i, t in enumerate(task_order)]
    col_labels = task_order
    df_matrix = pd.DataFrame(R_matrix, index=row_labels, columns=col_labels)

    # Főátló feletti értékek elfedése
    mask = np.triu(np.ones_like(R_matrix, dtype=bool), k=1)

    fig, ax = plt.subplots(figsize=(8.5, 6.8), dpi=300)
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    heatmap = sns.heatmap(
        df_matrix,
        mask=None,
        annot=True,
        fmt=".4f", # Hány tizedesjegy pontosság legyen
        annot_kws={"size": 11, "weight": "bold"},
        cmap="Blues",
        vmin=0.0,
        vmax=100.0,
        linewidths=1.2,
        linecolor="white",
        cbar_kws={
            "label": "Test Accuracy (%)",
            "shrink": 0.88,
            "pad": 0.04,
            "ticks": np.arange(0, 101, 10),
        },
        ax=ax,
    )

    # Szövegszín korrekció: sötét kék háttéren fehér, világoson fekete
    for text in heatmap.texts:
        try:
            val = float(text.get_text())
            text.set_color("black" if val < 50.0 else "white")
        except ValueError:
            pass

    ax.set_title(
        f"T x T Performance Matrix - {experiment_name}",
        fontsize=13,
        fontweight="bold",
        color="black",
        pad=18,
    )
    ax.set_xlabel(
        "Evaluated Task (Test Set)",
        fontsize=11,
        fontweight="semibold",
        color="black",
        labelpad=12,
    )
    ax.set_ylabel(
        "Training Phase (After Task)",
        fontsize=11,
        fontweight="semibold",
        color="black",
        labelpad=12,
    )

    ax.tick_params(colors="black", labelsize=10)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", color="black")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, color="black")

    cbar = heatmap.collections[0].colorbar
    cbar.ax.tick_params(color="black", labelcolor="black", labelsize=10)
    cbar.set_label(
        "Test Accuracy (%)",
        fontsize=10.5,
        fontweight="semibold",
        color="black",
        labelpad=10,
    )

    plt.tight_layout()

    # Mentés biztonságos abszolút útvonalra és tiszta paraméterezéssel
    img_path = os.path.join(RESULT_DIR, f"{experiment_name}_heatmap.png")
    fig.savefig(img_path, dpi=300, facecolor="white", bbox_inches="tight")
    # plt.show()

    print(f"Ábra sikeresen elmentve: {img_path}")

## Javasolt mérőszámok kiszámítása és mentése JSON fájlba

### 1. Végső Átlagos Pontosság (`final_average_accuracy` – $A_T$)
A teljes tanulási folyamat legvégén (a $T$-edik feladat után) a modell tesztkészleteken mért pontosságainak számtani átlaga. Megadja a modell globális, végső megtartott tudásszintjét:
$$A_T = \frac{1}{T} \sum_{j=1}^{T} R_{T,j}$$
* **Értelmezés:** A **continous learning-ben** a cél ennek maximalizálása. Sequential fine-tuning esetén a korábbi feladatok felejtése miatt ez az érték drasztikusan elmarad a Joint-training (felső referencia) szintjétől.

---

### 2. Lépésenkénti Átlagos Pontosság (`step_average_accuracies` – $A_i$)
Az $i$-edik feladat megtanulása után az **addig látott feladatok** ($j \in \{1, \dots, i\}$) átlagos pontossága:
$$A_i = \frac{1}{i} \sum_{j=1}^{i} R_{i,j} \quad (i = 1, \dots, T)$$
* **Értelmezés:** Nyomon követi a rendszer átlagos tudásának trajektóriáját az idővonal mentén. Megmutatja, hogy egy új feladat belépése mennyire veti vissza az addig felépített átlagos teljesítményt.

---

### 3. Visszamenőleges Transzfer (`backward_transfer` – $\text{BWT}$)
Számszerűsíti, hogy a későbbi feladatok ($\tau_{j+1}, \dots, \tau_T$) elsajátítása hogyan befolyásolta a korábbi feladatok ($\tau_j$) megtartott tudását a saját tanításukkor elért **csúcs pontossághoz** ($R_{j,j}$) viszonyítva:
$$\text{BWT} = \frac{1}{T-1} \sum_{j=1}^{T-1} \left( R_{T,j} - R_{j,j} \right)$$
* **Feladatonkénti BWT (`bwt_per_task`):**
  $$\text{BWT}_j = R_{T,j} - R_{j,j} \quad (j = 1, \dots, T-1)$$
* **Értelmezés:** 
  * $\text{BWT} < 0$: **Catastrophic forgetting**. Az új feladatok tanulása rontotta a korábbi feladatokon elért pontosságot. Minél nagyobb a negatív érték annál jobban felejtett a modell
  * $\text{BWT} = 0$: Tökéletes megőrizte a korábbi tudását, de az új feladatok nem segítették a régi feladatok teljesítményét. 
  * $\text{BWT} > 0$: **Pozitív visszamenőleges transzfer**. Az újabb feladatok tanulása során elsajátított közös reprezentációk még javították is a korábbi feladatok általánosítását.

---

### 4. Átlagos Felejtés (`average_forgetting` – $F$)


Minden korábbi feladatra vizsgálja a tanítás során elért maximális teljesítmény és a legvégső lépésben ($T$) maradt pontosság közötti veszteséget, pozitív mérőszámként definiálva:
$$F = \frac{1}{T-1} \sum_{j=1}^{T-1} f_j$$
ahol a $j$-edik feladat felejtése (`forgetting_per_task`):
$$f_j = \max_{k \in \{j, \dots, T-1\}} R_{k,j} - R_{T,j}$$
* **BWT vs Average Forgetting:** Míg a BWT azt vizsgálja, hogy a folyamat legvégén hogyan áll a múltbeli tudás az eredetihez képest, addig az Average Forgetting azt számszerűsíti, hogy a feladatok a tanulási folyamat során maximális teljesítményükhöz képest átlagosan mennyit romlottak.


---

### 5. Előrefelé Transzfer – Tanítás Előtt (`forward_transfer_pre` – $\text{FWT}_{\text{pre}}$)
Azt vizsgálja, hogy a modell hogyan teljesít egy adott feladaton **közvetlenül annak betanítása előtt**, kizárólag a korábban látott feladatokból származó tudás (backbone) és még a véletlenszerű kimeneti fej alapján: 
$$\text{FWT}_{\text{pre}} = \frac{1}{T-1} \sum_{j=2}^{T} R_{j-1, j}$$
* **Feladatonkénti tanítás előtti transzfer (`fwt_pre_per_task`):**
  $$\text{FWT}_{\text{pre}, j} = R_{j-1, j} \quad (j = 2, \dots, T)$$
* **Értelmezés:** A mátrix főátló feletti közvetlen szomszédos elemeinek átlaga. Multi-head architektúrában a $j$-edik feladat kimeneti feje ebben a fázisban még véletlenszerű inicializálású, így ez az érték a még nem kalibrált fej és a korábbi feladatokon edzett reprezentációk együttes zéró-shot bázisát mutatja.

---

### 6. Előrefelé Transzfer – Single-Task Referenciával (`forward_transfer_single` – $\text{FWT}_{\text{single}}$)
Azt vizsgálja, hogy a folyamatos tanulás során az előzetesen megszerzett reprezentációk segítik-e a $j$-edik feladat jobb megtanulását ahhoz képest, mintha azt teljesen izoláltan, nulláról (Single-task Oracle) tanultuk volna meg:
$$\text{FWT}_{\text{single}} = \frac{1}{T-1} \sum_{j=2}^{T} \left( R_{j,j} - b_{\text{single}, j} \right)$$
* **Feladatonkénti FWT (`fwt_single_per_task`):**
  $$\text{FWT}_{\text{single}, j} = R_{j,j} - b_{\text{single}, j} \quad (j = 2, \dots, T)$$
  ahol $b_{\text{single}, j}$ a $\tau_j$ feladaton izoláltan, önállóan betanított modell tesztpontossága.
* **Értelmezés:**
  * $\text{FWT}_{\text{single}} > 0$: **Pozitív előrehatás**. A korábbi feladatok tudásbázisa előnyt biztosított az új feladat elsajátításakor.
  * $\text{FWT}_{\text{single}} < 0$: **Negatív interferencia / plaszticitásvesztés**. A korábbi feladatokhoz merevedett súlyok gátolták az új feladatra való optimális ráállást.
  * Ha a referenciavektor ($b_{\text{single}}$) nincs megadva, az értéke `null`, a feladatonkénti szótár pedig `{}`.

In [5]:
RESULT_DIR = os.path.abspath("results")
os.makedirs(RESULT_DIR, exist_ok=True)

def compute_and_save_cl_metrics(R_matrix, task_order, seed=42, experiment_name="sequential_baseline", b_single_dict=None):
    """
    A T x T mátrix vizualizációjától teljesen függetlenül:
    1. Kiszámítja az összes kötelező CL metrikát (Final Acc, Step Acc, BWT, Forgetting, FWT).
    2. Elmenti az eredményeket strukturált JSON és összefoglaló CSV formátumban.
    3. Kiír egy áttekinthető szöveges összegzést a konzolra.
    """
    T = len(task_order)
    R = np.array(R_matrix, dtype=float)

    # 1. Final Average Accuracy (A_T) - utolsó sor átlaga
    final_avg_acc = float(np.mean(R[T-1, :]))

    # 2. Lépésenkénti Average Accuracy (A_i) - i-edik lépés után az addigi taskok átlaga
    step_avg_acc = {}
    for i in range(T):
        step_avg_acc[f"Step_{i+1}_{task_order[i]}"] = round(float(np.mean(R[i, :i+1])), 4)

    # 3. Backward Transfer (BWT) - utolsó sor vs. főátló az első T-1 taskra
    bwt_per_task = {}
    for j in range(T - 1):
        diff = R[T-1, j] - R[j, j]
        bwt_per_task[task_order[j]] = round(float(diff), 4)
    bwt = float(np.mean(list(bwt_per_task.values()))) if (T > 1) else 0.0

    # 4. Average Forgetting (F) - csúcsteljesítmény vs. utolsó sor
    forgetting_per_task = {}
    for j in range(T - 1):
        peak = np.max(R[j:T-1, j])
        f_j = peak - R[T-1, j]
        forgetting_per_task[task_order[j]] = round(float(f_j), 4)
    avg_forgetting = float(np.mean(list(forgetting_per_task.values()))) if (T > 1) else 0.0

    # 5. Forward Transfer (FWT)
    # A) Tanítás előtti (zero-shot) FWT a főátló feletti közvetlen szomszédokból
    fwt_pre_per_task = {}
    for j in range(1, T):
        fwt_pre_per_task[task_order[j]] = round(float(R[j-1, j]), 4)
    fwt_pre = float(np.mean(list(fwt_pre_per_task.values()))) if (T > 1) else 0.0

    # B) Single-task referencia alapú FWT (ha rendelkezésre áll)
    fwt_single = None
    fwt_single_per_task = {}
    if b_single_dict is not None:
        diffs = []
        for j in range(1, T):
            t_name = task_order[j]
            if t_name in b_single_dict:
                d = R[j, j] - b_single_dict[t_name]
                fwt_single_per_task[t_name] = round(float(d), 4)
                diffs.append(d)
        if len(diffs) > 0:
            fwt_single = round(float(np.mean(diffs)), 4)

    # Összesített dictionary mentéshez
    metrics_summary = {
        "experiment_name": experiment_name,
        "seed": seed,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "task_order": task_order,
        "final_average_accuracy": round(final_avg_acc, 4),
        "backward_transfer": round(bwt, 4),
        "average_forgetting": round(avg_forgetting, 4),
        "forward_transfer_pre": round(fwt_pre, 4),
        "forward_transfer_single": fwt_single,
        "step_average_accuracies": step_avg_acc,
        "bwt_per_task": bwt_per_task,
        "forgetting_per_task": forgetting_per_task,
        "fwt_pre_per_task": fwt_pre_per_task,
        "fwt_single_per_task": fwt_single_per_task
    }

    # 1. Mentés JSON fájlba (teljes strukturált riport)
    json_path = os.path.join(RESULT_DIR, f"{experiment_name}_seed{seed}_metrics.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(metrics_summary, f, indent=4, ensure_ascii=False)

    # 3. Áttekinthető szöveges kiírás a konzolra
    # print("\n" + "=" * 55)
    # print(f" CONTINUAL LEARNING METRIKÁK (Seed: {seed})")
    # print("=" * 55)
    # print(f" Final Average Accuracy (A_T): {final_avg_acc:.2f}%")
    # print(f" Backward Transfer (BWT):      {bwt:.2f}%")
    # print(f" Average Forgetting (F):       {avg_forgetting:.2f}%")
    # print(f" Forward Transfer (Zero-shot): {fwt_pre:.2f}%")
    # if fwt_single is not None:
    #     print(f" Forward Transfer (Single-ref):{fwt_single:.2f}%")
    # print("-" * 55)
    # print("Lépésenkénti átlagos pontosságok (A_i):")
    # for step, acc in step_avg_acc.items():
    #     print(f"  {step}: {acc:.2f}%")
    # print("-" * 55)
    # print("Felejtés feladatonként (Forgetting):")
    # for t_name, f_val in forgetting_per_task.items():
    #     print(f"  {t_name}: {f_val:.2f}%")
    # print("=" * 55)
    print(f"Eredmények elmentve:")
    print(f"  -> JSON: {json_path}")

    return metrics_summary

Res-Net-18 mert ez van a MedMNIST hivatlos kutatási keretrendszerében

In [4]:
def create_model(num_classes):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)


In [5]:
class MultiHeadResNet18(nn.Module):
    def __init__(self, task_classes_dict):
        """
        task_classes_dict: Egy szótár, ami megadja, hogy melyik taskhoz hány osztály tartozik.
                           Pl. {'pathmnist': 9, 'dermamnist': 7, 'octmnist': 4, 'pneumoniamnist': 2, 'organamnist': 11}
        """
        super(MultiHeadResNet18, self).__init__()

        # 1. Közös 2D Backbone (ResNet-18, weights=None a tiszta FWT referenciához)
        base_resnet = models.resnet18(weights=None)

        # Levesszük az eredeti fc réteget, hogy csak a konvolúciós feature extractor maradjon
        self.backbone = nn.Sequential(*list(base_resnet.children())[:-1])
        in_features = base_resnet.fc.in_features # általában 512

        # 2. Task-specifikus kimeneti fejek (Task-specific heads) nn.ModuleDict-ben
        self.heads = nn.ModuleDict()
        for task_name, num_classes in task_classes_dict.items():
            self.heads[task_name] = nn.Linear(in_features, num_classes)

    def forward(self, x, task_name):
        # 1. Átküldjük a képet a közös backbone-on
        features = self.backbone(x)
        features = torch.flatten(features, 1) # [Batch_size, 512]

        # 2. Kiválasztjuk a task_name (task_id) alapján a megfelelő fejet
        if task_name not in self.heads:
            raise ValueError(f"Ismeretlen task név: {task_name}")

        out = self.heads[task_name](features)
        return out

Kiértékeli a modell teljesítményét

In [8]:
def evaluate_detailed(model, loader, task_type, criterion, eval_task=None):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            # outputs = model(images,task_name=eval_task)

            if hasattr(model, 'heads') and eval_task is not None:
                outputs = model(images, task_name=eval_task) # Itt adom át az éppen futtatott modell nevét ahhoz hogy a backboneból a megfelelő fejet találja meg
            else:
                outputs = model(images)

            if task_type == 'multi-label, binary-class':
                labels = labels.to(device).float()
                loss = criterion(outputs, labels)
                probs = torch.sigmoid(outputs)
                preds = (probs > 0.5).float()
            else:
                labels = labels.squeeze(-1).long().to(device)
                loss = criterion(outputs, labels)
                probs = F.softmax(outputs, dim=1)
                preds = outputs.argmax(dim=1)

            total_loss += loss.item() * images.size(0)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)

    # Biztonságos metrikaszámítás feladattípus szerint
    if task_type == 'multi-label, binary-class':
        accuracy = (all_preds == all_labels).mean() * 100.0
        precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)
        try:
            roc_auc = roc_auc_score(all_labels, all_probs, average='macro')
        except ValueError:
            roc_auc = 0.0
        cm = None  # Multi-label esetén a konfúziós mátrix nem egyértelmű mátrixként adódik
    else:
        accuracy = (all_preds == all_labels).mean() * 100.0
        precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)
        cm = confusion_matrix(all_labels, all_preds)
        try:
            n_classes = all_probs.shape[1]
            y_true_onehot = np.eye(n_classes)[all_labels]
            roc_auc = roc_auc_score(y_true_onehot, all_probs, multi_class='ovr', average='macro')
        except ValueError:
            roc_auc = 0.0

    metrics_dict = {
        "loss": round(avg_loss,4),
        "accuracy": float(round(accuracy,4)),
        "precision": round(precision * 100.0,4),
        "recall": round(recall * 100.0,4),
        "f1_score": round(f1 * 100.0,4),
        "roc_auc": round(roc_auc,4)
    }

    return metrics_dict

Egyszeri folyamatos tanulás

In [9]:
def train_single_task(dataset_name, epochs=3, batch_size=128, lr=1e-3, max_train_samples=2000, max_eval_samples=500):
    seed=set_seed(42)
    print(f"\n==========================================")
    print(f"Adathalmaz: {dataset_name.upper()}")
    train_loader, val_loader, test_loader, info = get_dataloaders(
        dataset_name, batch_size, max_train_samples, max_eval_samples
    )

    task_type = info['task']
    num_classes = len(info['label'])
    print(f"Feladat: {task_type} | Osztályok száma: {num_classes}")
    print(f"Használt mintaszám -> Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}, Test: {len(test_loader.dataset)}")

    model = create_model(num_classes)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss() if task_type == 'multi-label, binary-class' else nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_weights = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for images, labels in train_loader:
            images = images.to(device)
            optimizer.zero_grad()
            outputs = model(images)

            labels = labels.to(device).float() if task_type == 'multi-label, binary-class' else labels.squeeze(-1).long().to(device)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)
        val_metrics = evaluate_detailed(model, val_loader, task_type, criterion)
        val_loss=val_metrics["loss"]
        val_acc=val_metrics["accuracy"]
        # display(val_metrics)
        print(f"Epoch [{epoch}/{epochs}] - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_weights)
    test_metrics = evaluate_detailed(model, test_loader, task_type, criterion)
    test_loss=test_metrics["loss"]
    test_acc=test_metrics["accuracy"]
    # display(test_metrics)
    # print(f"--> VÉGSŐ TESZT PONTOSSÁG ({dataset_name.upper()}): {test_acc:.2f}%")
    # save_results_to_file(test_metrics)
    return model, test_acc

## 2. Munkacsomag: Előzetes Task Hasonlóság (Similarity) Mérése

In [ ]:
def compute_task_similarity_matrix(task_list, num_samples=500,seed=42):
    """
    Kiszámítja az adathalmazok közötti távolságot egy előtanított ResNet-18
    feature centroidjainak koszinusz-távolsága alapján (nem használ tesztadatot).
    """
    set_seed(seed)
    print(f"--- Előzetes Task Similarity elemzés futtatása Seed: {seed}---")
    feature_extractor = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device)
    feature_extractor.fc = nn.Identity()
    feature_extractor.eval()

    centroids = {}
    with torch.no_grad():
        for task_name in task_list:
            train_loader, _, _, _ = get_dataloaders(task_name, batch_size=64, max_train_samples=num_samples)
            features_list = []
            for images, _ in train_loader:
                images = images.to(device)
                feats = feature_extractor(images)
                features_list.append(feats.cpu())
            all_feats = torch.cat(features_list, dim=0)
            centroid = torch.mean(all_feats, dim=0)
            centroid = centroid / torch.norm(centroid, p=2)  # L2 normalizálás
            centroids[task_name] = centroid

    num_tasks = len(task_list)
    dist_matrix = np.zeros((num_tasks, num_tasks))
    for i in range(num_tasks):
        for j in range(num_tasks):
            t1, t2 = task_list[i], task_list[j]
            # Koszinusz távolság: 1 - cosine_similarity
            cos_sim = torch.dot(centroids[t1], centroids[t2]).item()
            dist_matrix[i, j] = abs(round(1.0 - cos_sim, 4))

    df_dist = pd.DataFrame(dist_matrix, index=task_list, columns=task_list)
    df_dist.to_csv("results/task_distance_matrix.csv")

    plt.close("all")
    plt.rcParams.update(plt.rcParamsDefault)
    sns.set_theme(style="white")

    # Hasonlósági hőtérkép rajzolása
    fig, ax = plt.subplots(figsize=(6.5, 5.2), dpi=300)
    sns.heatmap(df_dist, annot=True, fmt=".3f", cmap="Blues_r", ax=ax,
                cbar_kws={'label': 'Koszinusz távolság (kisebb = hasonlóbb)'})
    ax.set_title("Előzetes Task Távolság Mátrix (Feature Cosine Distance)", fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig("results/task_similarity_heatmap.png", dpi=300)
    plt.show()

    print("Task távolságok elmentve: results/task_distance_matrix.csv és results/task_similarity_heatmap.png")
    return df_dist


# Futtatás az 5 adathalmazra
task_pool = ['pathmnist', 'dermamnist', 'octmnist', 'pneumoniamnist', 'organamnist']
dist_df = compute_task_similarity_matrix(task_pool, num_samples=500)

## Task similarity Wasserstein-távolságot (Earth Mover's Distance - EMD)

In [ ]:
def compute_pixel_wasserstein_matrix(task_list, num_samples=500, seed=42):
    """
    Kiszámítja az adathalmazok közötti távolságot a nyers pixel-intenzitás eloszlások
    Wasserstein-távolsága (Earth Mover's Distance) alapján.
    Teljesen modellfüggetlen referencia a reprezentációs távolság mellé.
    """
    set_seed(seed)
    print(f"--- Nyers Pixel Wasserstein-távolság elemzés (Seed: {seed}) ---")

    pixel_distributions = {}

    # 1. Pixelértékek gyűjtése feladatonként az első num_samples mintából
    for task_name in task_list:
        info = INFO[task_name]
        DataClass = getattr(medmnist, info['python_class'])

        # Egységes [0, 1] tartományra skálázott ToTensor normalizálás nélkül a valós intenzitásokhoz
        transform = transforms.ToTensor()
        dataset = DataClass(split='train', transform=transform, download=True, as_rgb=True)

        if num_samples is not None and num_samples < len(dataset):
            dataset = Subset(dataset, list(range(num_samples)))

        loader = DataLoader(dataset, batch_size=64, shuffle=False)

        task_pixels = []
        for images, _ in loader:
            # images alakja: [B, 3, 28, 28] -> lapítjuk 1D tömbbé
            task_pixels.append(images.numpy().ravel())

        pixel_distributions[task_name] = np.concatenate(task_pixels)

    # 2. Páronkénti 1D Wasserstein-távolság kiszámítása
    num_tasks = len(task_list)
    dist_matrix = np.zeros((num_tasks, num_tasks))

    for i in range(num_tasks):
        for j in range(num_tasks):
            t1, t2 = task_list[i], task_list[j]
            # Két empirikus eloszlás közötti EMD távolság
            dist = wasserstein_distance(pixel_distributions[t1], pixel_distributions[t2])
            dist_matrix[i, j] = round(float(dist), 4)

    df_dist = pd.DataFrame(dist_matrix, index=task_list, columns=task_list)

    # Mentés CSV fájlba
    csv_path = os.path.join(RESULT_DIR, "task_pixel_wasserstein_matrix.csv")
    df_dist.to_csv(csv_path)

    # 3. Publikációs minőségű hőtérkép rajzolása és mentése
    plt.close("all")
    plt.rcParams.update(plt.rcParamsDefault)
    sns.set_theme(style="white")

    fig, ax = plt.subplots(figsize=(7.2, 5.8), dpi=300)
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    heatmap = sns.heatmap(
        df_dist,
        annot=True,
        fmt=".3f",
        annot_kws={"size": 10, "weight": "bold"},
        cmap="Greens_r",  # Zöld skála a ResNet kékjétől való vizuális megkülönböztetéshez
        linewidths=1.0,
        linecolor="white",
        ax=ax,
        cbar_kws={"label": "Wasserstein-távolság (EMD) (kisebb = hasonlóbb)", "shrink": 0.88, "pad": 0.04}
    )

    # Szövegszínek kontrasztosítása
    max_val = df_dist.values.max()
    for text in heatmap.texts:
        try:
            val = float(text.get_text())
            text.set_color("black" if val > (max_val * 0.45) else "white")
        except ValueError:
            pass

    ax.set_title("Nyers Pixel Intenzitás Távolság Mátrix (Wasserstein EMD)", fontsize=11, fontweight="bold", color="black", pad=16)
    ax.tick_params(colors="black", labelsize=10)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", color="black")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, color="black")

    cbar = heatmap.collections[0].colorbar
    cbar.ax.tick_params(color="black", labelcolor="black", labelsize=9)
    cbar.set_label("Wasserstein-távolság (kisebb = hasonlóbb)", fontsize=10, fontweight="semibold", color="black", labelpad=10)

    plt.tight_layout()
    img_path = os.path.join(RESULT_DIR, "task_pixel_wasserstein_heatmap.png")
    fig.savefig(img_path, dpi=300, facecolor="white", bbox_inches="tight")
    plt.show()

    print(f"Eredmények elmentve:")
    print(f"  -> CSV: {csv_path}")
    print(f"  -> PNG: {img_path}")

    return df_dist

# Futtatás az 5 adathalmazra
task_pool = ['pathmnist', 'dermamnist', 'octmnist', 'pneumoniamnist', 'organamnist']
pixel_dist_df = compute_pixel_wasserstein_matrix(task_pool, num_samples=500, seed=42)

In [86]:
# 1. PathMNIST (~90 000 train minta)
model_path_full, acc_path_full = train_single_task('pathmnist', epochs=1, max_train_samples=None, max_eval_samples=None)

# 2. DermaMNIST (~7 000 train minta)
model_derma, acc_derma = train_single_task('dermamnist', epochs=1, max_train_samples=None, max_eval_samples=None)

# 3. OCTMNIST (~97 000 train minta)
model_oct, acc_oct = train_single_task('octmnist', epochs=1, max_train_samples=None, max_eval_samples=None)

# 4. PneumoniaMNIST (~4 700 train minta)
model_pneumonia, acc_pneumonia = train_single_task('pneumoniamnist', epochs=1, max_train_samples=None,
                                                   max_eval_samples=None)

# 5. Organamnist (~34561 train minta)
model_organ, acc_organ = train_single_task('organamnist', epochs=1, max_train_samples=None, max_eval_samples=None)

single_task_dict = {
    'pathmnist': acc_path_full,
    'dermamnist': acc_derma,
    'octmnist': acc_oct,
    'pneumoniamnist': acc_pneumonia,
    'organamnist': acc_organ,
}

# (Opcionális, de ajánlott) Mentsd el JSON-be, hogy ne kelljen újra futtatni:
with open("results/single_task_reference.json", "w") as f:
    json.dump(single_task_dict, f, indent=4)



Adathalmaz: PATHMNIST
Feladat: multi-class | Osztályok száma: 9
Használt mintaszám -> Train: 89996, Val: 10004, Test: 7180
Epoch [1/1] - Train Loss: 0.6731 | Val Loss: 0.8850 | Val Acc: 66.26%

Adathalmaz: DERMAMNIST
Feladat: multi-class | Osztályok száma: 7
Használt mintaszám -> Train: 7007, Val: 1003, Test: 2005
Epoch [1/1] - Train Loss: 0.9400 | Val Loss: 0.7201 | Val Acc: 70.09%

Adathalmaz: OCTMNIST
Feladat: multi-class | Osztályok száma: 4
Használt mintaszám -> Train: 97477, Val: 10832, Test: 1000
Epoch [1/1] - Train Loss: 0.5407 | Val Loss: 0.4066 | Val Acc: 85.61%

Adathalmaz: PNEUMONIAMNIST
Feladat: binary-class | Osztályok száma: 2
Használt mintaszám -> Train: 4708, Val: 524, Test: 624
Epoch [1/1] - Train Loss: 0.1968 | Val Loss: 0.5091 | Val Acc: 86.13%

Adathalmaz: ORGANAMNIST
Feladat: multi-class | Osztályok száma: 11
Használt mintaszám -> Train: 34561, Val: 6491, Test: 17778
Epoch [1/1] - Train Loss: 0.2906 | Val Loss: 0.1704 | Val Acc: 96.19%


Tanítás


In [ ]:
CURRENT_SEED=set_seed(42)
# 1. Előre definiált task sorrend és osztályszámaik
task_order = ['pathmnist', 'dermamnist', 'octmnist', 'pneumoniamnist', 'organamnist']
task_classes = {
    'pathmnist': 9,
    'dermamnist': 7,
    'octmnist': 4,
    'pneumoniamnist': 2,
    'organamnist': 11
}

# 2. Példányosítjuk a Multi-head modellt
model = MultiHeadResNet18(task_classes).to(device)

# 3. T x T Performance Mátrix inicializálása
T = len(task_order)
R_matrix = np.zeros((T, T))

# 4. Külső ciklus: Végigmegyünk a feladatokon a megadott sorrendben
for i, train_task in enumerate(task_order):
    print(f"\n==========================================")
    print(f"Fázis [{i+1}/{T}]: Tanítás a(z) {train_task.upper()} adathalmazon")
    print(f"==========================================s")

    # Letöltjük az aktuális task adagolóit (train, val, test)
    train_loader, val_loader, test_loader, info = get_dataloaders(train_task, batch_size=128)
    task_type = info['task']
    criterion = nn.BCEWithLogitsLoss() if task_type == 'multi-label, binary-class' else nn.CrossEntropyLoss()

    # Optimizer (csak a modell paramétereit frissítjük)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Egyszerű tanító ciklus (early stopping a validation szetten)
    best_val_acc = 0.0
    epochs = 1 # Teszthez 5, éles futáshoz 15-20 epoch

    for epoch in range(1, epochs + 1):
        model.train()
        for images, labels in train_loader:
            images = images.to(device)
            optimizer.zero_grad()

            # Forward a megfelelő task_name átadásával a Multi-head modellnek
            outputs = model(images, task_name=train_task)

            if task_type == 'multi-label, binary-class':
                labels = labels.to(device).float()
            else:
                labels = labels.squeeze(-1).long().to(device)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # Validation ellenőrzés
        val_metrics = evaluate_detailed(model, val_loader, task_type, criterion, eval_task=train_task)
        val_acc=val_metrics["accuracy"]
        print(f"Epoch {epoch}/{epochs} - Val Acc ({train_task}): {val_acc:.2f}%")

    # --- 5. KIÉRTÉKELÉS AZ ÖSSZES EDDIG LÁTOTT TASKON (Mátrix kitöltése) ---
    # print(f"\n-> Kiértékelés az összes taskokon a(z) {train_task} tanítása után:")
    model.eval()
    with torch.no_grad():
        for j,eval_task in enumerate(task_order): # Csak az eddig látott taskokat (vagy az összeset FWT-hez) teszteljük
            # Betöltjük az eval_task teszt loaderét
            _, _, eval_test_loader, eval_info = get_dataloaders(eval_task, batch_size=128)
            eval_type = eval_info['task']
            eval_crit = nn.BCEWithLogitsLoss() if eval_type == 'multi-label, binary-class' else nn.CrossEntropyLoss()

            test_metrics = evaluate_detailed(model, eval_test_loader, eval_type, eval_crit, eval_task=eval_task)
            test_acc=test_metrics["accuracy"]

            R_matrix[i, j] = test_acc
            # print(f"   Test Acc on {eval_task}: {test_acc:.2f}%")

print("\n=== VÉGSŐ T x T PERFORMANCE MÁTRIX ===")
save_publication_matrix(R_matrix=R_matrix,task_order=task_order, experiment_name=f"sequential_baseline{CURRENT_SEED}")


# --- 2. AZ ÚJ METRIKÁK KISZÁMÍTÁSA ÉS KÜLÖN KEZELÉSE ---
metrics = compute_and_save_cl_metrics(
    R_matrix=R_matrix,
    task_order=task_order,
    seed=CURRENT_SEED,
    experiment_name="sequential_baseline",
    b_single_dict=single_task_dict # Ha később lesz single-task referenciád, ide átadható
)

## 2. Munkacsomag: Task Order Kísérletek (4 sorrend összehasonlítása)

In [98]:
# 1. Kísérleti sorrendek definíciója
def generate_random_order(task_pool, seed):
    rng = random.Random(seed)
    return rng.sample(task_pool, len(task_pool))

base_tasks = ['pathmnist', 'dermamnist', 'octmnist', 'pneumoniamnist', 'organamnist']

experiment_orders = {
    "similar_first": ['dermamnist', 'pathmnist', 'organamnist', 'pneumoniamnist', 'octmnist'],

    "dissimilar_alternating": ['dermamnist', 'octmnist', 'pathmnist', 'pneumoniamnist', 'organamnist'],

    "random_order_1": generate_random_order(base_tasks, seed=101),

    "random_order_2": generate_random_order(base_tasks, seed=202)
}
# Tesztelés miatt kellett csak
# for name, order in experiment_orders.items():
#     print(f"[{name}]: {' -> '.join(order)}")

task_classes = {
    'pathmnist': 9,
    'dermamnist': 7,
    'octmnist': 4,
    'pneumoniamnist': 2,
    'organamnist': 11
}

# Ha már létezik korábbról mentett single-task referencia, betöltjük:
single_ref_path = "results/single_task_reference.json"
b_single_dict = None
if os.path.exists(single_ref_path):
    with open(single_ref_path, "r") as f:
        b_single_dict = json.load(f)

order_summary_rows = []
CURRENT_SEED = set_seed(42)
epochs = 1 # Gyors tesztfutáshoz 1, élesben növelhető

for exp_name, current_order in experiment_orders.items():
    print(f"\n==================================================")
    print(f" FUTTATÁS: {exp_name.upper()}")
    print(f" Sorrend: {' -> '.join(current_order)}")
    print(f"==================================================")

    # Friss modell példányosítása az adott sorrendhez
    model = MultiHeadResNet18(task_classes).to(device)
    T = len(current_order)
    R_matrix = np.zeros((T, T))

    for i, train_task in enumerate(current_order):
        train_loader, val_loader, _, info = get_dataloaders(train_task, batch_size=128)
        task_type = info['task']
        criterion = nn.BCEWithLogitsLoss() if task_type == 'multi-label, binary-class' else nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

        # Tanítás
        for epoch in range(1, epochs + 1):
            model.train()
            for images, labels in train_loader:
                images = images.to(device)
                optimizer.zero_grad()
                outputs = model(images, task_name=train_task)
                labels = labels.to(device).float() if task_type == 'multi-label, binary-class' else labels.squeeze(-1).long().to(device)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

        # Kiértékelés az ÖSSZES feladaton a sorrend szerint (TxT mátrix sorának kitöltése)
        model.eval()
        with torch.no_grad():
            for j, eval_task in enumerate(current_order):
                _, _, eval_test_loader, eval_info = get_dataloaders(eval_task, batch_size=128)
                eval_type = eval_info['task']
                eval_crit = nn.BCEWithLogitsLoss() if eval_type == 'multi-label, binary-class' else nn.CrossEntropyLoss()
                test_metrics = evaluate_detailed(model, eval_test_loader, eval_type, eval_crit, eval_task=eval_task)
                R_matrix[i, j] = test_metrics["accuracy"]

    # 1. Hőtérkép mentése
    save_publication_matrix(
        R_matrix=R_matrix,
        task_order=current_order,
        experiment_name=f"{exp_name}_seed{CURRENT_SEED}"
    )

    # 2. Metrikák számítása és mentése
    metrics = compute_and_save_cl_metrics(
        R_matrix=R_matrix,
        task_order=current_order,
        seed=CURRENT_SEED,
        experiment_name=exp_name,
        b_single_dict=b_single_dict
    )

    # 3. Összefoglaló sor készítése
    order_summary_rows.append({
        "Sorrend típus": exp_name,
        "Végső Átl. Pontosság (A_T) %": metrics["final_average_accuracy"],
        "Visszamenőleges Transzfer (BWT) %": metrics["backward_transfer"],
        "Átlagos Felejtés (F) %": metrics["average_forgetting"],
        "FWT (Single-ref) %": metrics["forward_transfer_single"]
    })

# Végső összehasonlító táblázat kiírása és mentése
df_order_comparison = pd.DataFrame(order_summary_rows)
df_order_comparison.to_csv("results/task_order_comparison.csv", index=False)

print("\n==================================================")
print(" 2. MUNKACSOMAG - TASK ORDER ÖSSZEHASONLÍTÁS:")
print("==================================================")
display(df_order_comparison)


 FUTTATÁS: SIMILAR_FIRST
 Sorrend: dermamnist -> pathmnist -> organamnist -> pneumoniamnist -> octmnist
Ábra sikeresen elmentve: C:\Munka\1. Féléves tárgyak\Fejlett gépi tanuló modellek\Projekt\results\similar_first_seed42_heatmap.png
Eredmények elmentve:
  -> JSON: C:\Munka\1. Féléves tárgyak\Fejlett gépi tanuló modellek\Projekt\results\similar_first_seed42_metrics.json

 FUTTATÁS: DISSIMILAR_ALTERNATING
 Sorrend: dermamnist -> octmnist -> pathmnist -> pneumoniamnist -> organamnist
Ábra sikeresen elmentve: C:\Munka\1. Féléves tárgyak\Fejlett gépi tanuló modellek\Projekt\results\dissimilar_alternating_seed42_heatmap.png
Eredmények elmentve:
  -> JSON: C:\Munka\1. Féléves tárgyak\Fejlett gépi tanuló modellek\Projekt\results\dissimilar_alternating_seed42_metrics.json

 FUTTATÁS: RANDOM_ORDER_1
 Sorrend: organamnist -> dermamnist -> octmnist -> pneumoniamnist -> pathmnist
Ábra sikeresen elmentve: C:\Munka\1. Féléves tárgyak\Fejlett gépi tanuló modellek\Projekt\results\random_order_1_seed

,Sorrend típus,Végső Átl. Pontosság (A_T) %,Visszamenőleges Transzfer (BWT) %,Átlagos Felejtés (F) %,FWT (Single-ref) %
0,similar_first,39.6452,-39.7385,39.7385,2.0194
1,dissimilar_alternating,49.8110,-32.8069,32.8069,7.8732
2,random_order_1,48.6148,-32.6067,32.6067,5.5359
3,random_order_2,43.2543,-38.4424,38.4424,4.4729


# 3. Munkacsomag: Continual-learning módszerek

Az 1. munkacsomagban felépített task-incremental protokollt és a 2. munkacsomag sorrend-kísérleteit most kiegészítjük a felejtést **aktívan csökkentő** eljárásokkal. A feladatkiírás kötelező minimuma:

| Módszer | Típus | Mit véd és mi az ára |
|---|---|---|
| Sequential fine-tuning | baseline | semmit nem véd, nincs többletköltség |
| Replay / exemplar memory | adatalapú | régi mintákat tárol, **memóriát** fogyaszt |
| EWC (Elastic Weight Consolidation) | regularizációs | a kritikus súlyokat rögzíti, **plaszticitást** áldoz |
| Partial freezing / adapterek | parameter isolation | fizikailag elkülöníti a súlyokat, **paramétert** fogyaszt |
| Joint training | felső referencia | nem continual, az elérhető maximumot méri |

### A közös dilemma: stabilitás és plaszticitás

* **Plaszticitás:** a háló azon képessége, hogy új ismereteket sajátítson el.
* **Stabilitás:** a modell képessége a korábban megszerzett tudás megőrzésére anélkül, hogy a súlyfrissítés elrontaná azt.

Mind a három védelmi módszer ugyanazt a kompromisszumot mozgatja: a stabilitást a plaszticitás rovására növeli. A mérés célja annak megállapítása, hogy adott memória- és paraméter-budget mellett melyik eljárás adja a legkedvezőbb cserearányt.

### Hol avatkozik be a kód

Mind a négy eljárás **ugyanabba a tanítási ciklusba** nyúl bele, három ponton:

1. a loss összeállításánál (replay, EWC),
2. az optimizer paraméterlistájánál (freezing, adapterek),
3. a task lezárása után (memória feltöltése, Fisher-mátrix frissítése).

Ezért egyetlen futtatókeretet (`run_continual_experiment`) írunk, amelyben a `method` kapcsoló választja ki az ágat. Így minden módszer ugyanazt a $T \times T$ mátrixot tölti ki és ugyanazokkal a metrikákkal értékelődik, tehát a számok közvetlenül összehasonlíthatók.

## 3.0 Közös konfiguráció és segédfüggvények

A reprodukálhatósági minimum miatt minden kísérlet konfigurációja géppel olvasható formában (JSON) mentésre kerül a nyers eredmények mellé.

**Eltérés a korábbi munkacsomagokhoz képest:** a validációs és teszt `DataLoader`-ekben a `drop_last` értéke itt `False`. Az 1-2. munkacsomagban `True` volt, ami eldobta az utolsó, nem teljes batchet, így például az OCTMNIST 1000 elemű tesztkészletéből csak 896, a PneumoniaMNIST 624 eleméből csak 512 minta került kiértékelésre. A hivatalos split teljes használatához a kiértékelésnél nem szabad mintát eldobni; a tanító loaderben a `drop_last=True` megmarad, mert az egyelemű batch a BatchNorm réteget elrontaná.

Emiatt a 3. munkacsomag `sequential` ága **újra lefuttatja a baseline-t** a javított protokollal, hogy a módszerek összehasonlítása azonos alapon történjen. A WP1 `sequential_baseline` eredményei ezért nem vethetők össze közvetlenül az itteni számokkal.

In [ ]:
import time

# ---------------------------------------------------------------------------
# 3. munkacsomag - közös konfiguráció
# ---------------------------------------------------------------------------
WP3_TASK_ORDER = ['pathmnist', 'dermamnist', 'octmnist', 'pneumoniamnist', 'organamnist']
WP3_TASK_CLASSES = {
    'pathmnist': 9,
    'dermamnist': 7,
    'octmnist': 4,
    'pneumoniamnist': 2,
    'organamnist': 11,
}

WP3_CONFIG = {
    "setting": "task-incremental continual learning, task-specifikus fejek",
    "backbone": "torchvision resnet18, weights=None",
    "task_order": WP3_TASK_ORDER,
    "task_classes": WP3_TASK_CLASSES,
    "seed": 42,
    "epochs_per_task": 1,        # tesztfutáshoz 1; éles méréshez 10-20 ajánlott
    "batch_size": 128,
    "optimizer": "Adam",
    "learning_rate": 1e-3,
    "augmentation": "nincs (ToTensor + Normalize(mean=0.5, std=0.5))",
    "train_drop_last": True,
    "eval_drop_last": False,     # WP1/WP2-ben True volt -> ott csonkult a teszthalmaz
    "replay": {
        "buffer_total_size": 2000,
        "sampling": "class_balanced",
        "allocation": "fix teljes keret, egyenlő elosztás az eddig látott taskok között",
    },
    "ewc": {
        "lambda": 5000.0,
        "fisher_batches": 50,
        "fisher_type": "empirikus diagonális Fisher a valódi címkék log-likelihoodjából",
        "accumulation": "online: a korábbi taskok Fisher-mátrixát összegezzük",
    },
    "adapter": {
        "module": "1x1 conv residual adapter, zero-init",
        "insertion": "layer1-layer4 stage-ek után",
        "backbone_frozen_after_task_index": 1,
    },
    "joint": {
        "strategy": "round-robin batching",
        "steps_per_epoch": "a leghosszabb train loader hossza",
    },
}


# ---------------------------------------------------------------------------
# Adatbetöltés gyorsítótárral
# ---------------------------------------------------------------------------
_WP3_DATA_CACHE = {}

def get_cl_data(dataset_name, batch_size=128, eval_drop_last=False):
    """
    A get_dataloaders továbbfejlesztett változata a 3. munkacsomaghoz.

    Két különbség:
      1. A val/test loader nem dobja el az utolsó részleges batchet, így a hivatalos
         teszthalmaz MINDEN mintáján értékelünk.
      2. Gyorsítótárazza a datasetet: a T x T mátrix kitöltése egyébként 25-ször
         építené újra ugyanazokat az adathalmazokat.

    A nyers train_dataset is visszaadásra kerül, mert a replay puffernek szüksége van rá.
    """
    key = (dataset_name, batch_size, eval_drop_last)
    if key in _WP3_DATA_CACHE:
        return _WP3_DATA_CACHE[key]

    info = INFO[dataset_name]
    DataClass = getattr(medmnist, info['python_class'])
    data_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])

    train_dataset = DataClass(split='train', transform=data_transform, download=True, as_rgb=True)
    val_dataset   = DataClass(split='val',   transform=data_transform, download=True, as_rgb=True)
    test_dataset  = DataClass(split='test',  transform=data_transform, download=True, as_rgb=True)

    bundle = {
        "train_loader": DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  drop_last=True),
        "val_loader":   DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, drop_last=eval_drop_last),
        "test_loader":  DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, drop_last=eval_drop_last),
        "train_dataset": train_dataset,
        "info": info,
        "n_train": len(train_dataset),
    }
    _WP3_DATA_CACHE[key] = bundle
    return bundle


def get_criterion(task_type):
    """A MedMNIST feladattípushoz tartozó veszteségfüggvény."""
    return nn.BCEWithLogitsLoss() if task_type == 'multi-label, binary-class' else nn.CrossEntropyLoss()


def prepare_labels(labels, task_type):
    """Címkék eszközre mozgatása és alakra hozása a feladattípus szerint."""
    if task_type == 'multi-label, binary-class':
        return labels.to(device).float()
    return labels.squeeze(-1).long().to(device)


def count_parameters(model, only_trainable=False):
    """Paraméterszámlálás a memória/compute trade-off dokumentálásához."""
    if only_trainable:
        return int(sum(p.numel() for p in model.parameters() if p.requires_grad))
    return int(sum(p.numel() for p in model.parameters()))


print("3. munkacsomag konfigurációja betöltve.")
print(f"  Sorrend: {' -> '.join(WP3_TASK_ORDER)}")
print(f"  Seed: {WP3_CONFIG['seed']} | Epoch/task: {WP3_CONFIG['epochs_per_task']} | LR: {WP3_CONFIG['learning_rate']}")

## 3.1 Replay / exemplar memory

A replay nem zárja le teljesen a korábbi feladatok adatforrását, hanem egy szigorúan korlátozott kapacitású külső memóriában (exemplar buffer, $\mathcal{M}$) reprezentatív mintákat őriz meg a múltbeli adateloszlásokból, és ezeket az új feladatok tanulása során periodikusan visszajátssza a hálózatnak.

A tanítási veszteség így két tagra bomlik:

$$\mathcal{L} = \underbrace{\mathcal{L}_{\tau_i}(x, y)}_{\text{aktuális feladat}} + \underbrace{\sum_{k < i} \mathcal{L}_{\tau_k}(x_\mathcal{M}, y_\mathcal{M})}_{\text{memóriából visszajátszott régi feladatok}}$$

A második tag miatt a régi feladatok gradiense újra eljut a közös gerinchálózathoz, tehát a backbone nem tud szabadon elmászni.

### Mintavételi stratégia

A jegyzet három lehetőséget vizsgál:

* **Uniform random sampling:** torzítatlan becslést ad az eredeti halmaz eloszlásáról, de előfordulhat, hogy szélsőértékek (outlierek) kerülnek a memóriába.
* **Herding (jellemzőátlag):** a modell osztályonként kiszámítja a belső reprezentációk átlagát (osztály-centroid), majd olyan mintákat választ, amelyek kumulatív átlaga a lehető legközelebb marad a centroidhoz.
* **Class-balanced reservoir sampling:** minden feladaton belül garantálja, hogy a pufferbe minden osztályból azonos számú minta kerüljön.

Az implementáció a **class-balanced** változatot használja, mert a MedMNIST feladatok erősen kiegyensúlyozatlanok (a DermaMNIST legnagyobb osztálya a minták kétharmadát teszi ki), és uniform mintavétellel a ritka osztályok kieshetnének a memóriából.

### Abszolút kapacitás és relatív arány

A feladatkiírás kifejezetten megköveteli, hogy a memória méretét **abszolút mintaszámban és a korábbi adathoz viszonyított arányban is** dokumentáljuk, különben a módszerek erőforrásigénye nem összehasonlítható:

$$p_i = \frac{|\mathcal{M}|}{\sum_{k \le i} N_k} \cdot 100\%$$

Ugyanaz a $K$ mintaszám drámaian eltérő információsűrűséget jelent a különböző feladatokon: $K = 100$ kép a PathMNIST 89 996 tanítómintájából $p \approx 0{,}11\%$, a PneumoniaMNIST 4 708 mintájából viszont $p \approx 2{,}12\%$. A fix méretű puffer tehát aszimmetrikus információsűrűséget hordoz.

Az implementáció a **teljes puffert** tartja fixen (`buffer_total_size = 2000`), és a kvótát egyenlően osztja szét az eddig látott feladatok között. Ahogy nő a feladatok száma, úgy szűkül a taskonkénti keret ($2000 \to 1000 \to 666 \to 500 \to 400$); a régi feladatok mintáit ilyenkor nyesni kell. Ez felel meg a kiírás „fixed-size replay buffer” követelményének.

> **Ismert kockázat (jegyzet 4. oldal):** a hálózat a memóriában lévő néhány száz képet sok epochon keresztül látja, ami túltanulást okozhat ezeken a mintákon, és az általánosítás rovására mehet.

In [ ]:
class ExemplarBuffer:
    """
    Fix TELJES méretű, class-balanced exemplar memória.

    A puffer teljes kapacitása (total_size) állandó marad a futás során: ahogy nő a
    látott feladatok száma, úgy csökken a feladatonkénti kvóta (total_size // n_task),
    és a régi feladatok tárolt mintáit lenyessük. Ez a feladatkiírás által kötelezően
    előírt "fixed-size replay buffer".

    A kiválasztás osztályonként kiegyensúlyozott: minden osztályból közel azonos számú
    mintát tartunk meg, hogy a ritka osztályok ne essenek ki a memóriából. A kiválasztott
    indexeket osztályok szerint körbejárva fűzzük össze, így a lista bármely kezdőszelete
    is közel kiegyensúlyozott marad - erre a későbbi nyeséskor van szükség.
    """

    def __init__(self, total_size=2000, seed=42):
        self.total_size = total_size
        self.rng = np.random.RandomState(seed)
        self.store = {}             # task_name -> {"x": tensor, "y": tensor}
        self.source_indices = {}    # task_name -> a kiválasztott dataset-indexek
        self.task_train_sizes = {}  # task_name -> az eredeti train halmaz mérete

    def _budget_per_task(self, n_tasks):
        return max(1, self.total_size // max(1, n_tasks))

    def _select_class_balanced(self, dataset, budget):
        labels = np.asarray(dataset.labels).reshape(len(dataset), -1)[:, 0]
        classes = np.unique(labels)
        per_class = int(np.ceil(budget / len(classes)))

        pools = []
        for c in classes:
            idx_c = np.where(labels == c)[0].copy()
            self.rng.shuffle(idx_c)
            pools.append(idx_c[:per_class].tolist())

        chosen = []
        for k in range(per_class):
            for pool in pools:
                if k < len(pool):
                    chosen.append(int(pool[k]))
        return chosen[:budget]

    def _materialize(self, dataset, indices):
        """A kiválasztott mintákat transzformált tenzorként tároljuk el."""
        xs, ys = [], []
        for idx in indices:
            x, y = dataset[idx]
            xs.append(x)
            ys.append(torch.as_tensor(np.asarray(y)))
        return torch.stack(xs), torch.stack(ys)

    def add_task(self, task_name, dataset):
        """Feladat lezárása: mintavétel a pufferbe, majd a kvóta újraosztása."""
        self.task_train_sizes[task_name] = len(dataset)
        n_tasks = len(self.store) + 1
        budget = self._budget_per_task(n_tasks)

        indices = self._select_class_balanced(dataset, budget)
        x, y = self._materialize(dataset, indices)
        self.store[task_name] = {"x": x, "y": y}
        self.source_indices[task_name] = indices

        # A korábbi feladatokat lenyessük az új kvótára -> a teljes méret fix marad
        for t in list(self.store.keys()):
            if t == task_name:
                continue
            if self.store[t]["x"].shape[0] > budget:
                self.store[t]["x"] = self.store[t]["x"][:budget]
                self.store[t]["y"] = self.store[t]["y"][:budget]
                self.source_indices[t] = self.source_indices[t][:budget]

    def sample(self, batch_size):
        """
        Vegyes minibatch a teljes memóriából, feladatonként csoportosítva.

        A csoportosítás azért kell, mert a multi-head forward hívásához tudni kell,
        melyik feladat fejét használjuk. A mintavétel a teljes pufferen egyenletes,
        tehát a nagyobb kvótájú feladatok arányosan többször szerepelnek.
        """
        if not self.store:
            return []

        tasks = list(self.store.keys())
        sizes = [self.store[t]["x"].shape[0] for t in tasks]
        total = int(sum(sizes))
        take = min(batch_size, total)

        flat = self.rng.choice(total, size=take, replace=False)
        offsets = np.cumsum([0] + sizes)

        batches = []
        for ti, t in enumerate(tasks):
            lo, hi = offsets[ti], offsets[ti + 1]
            sel = flat[(flat >= lo) & (flat < hi)] - lo
            if sel.size == 0:
                continue
            sel = torch.as_tensor(np.sort(sel), dtype=torch.long)
            batches.append((t, self.store[t]["x"][sel], self.store[t]["y"][sel]))
        return batches

    def memory_report(self, seen_tasks):
        """
        A memóriahasználat dokumentálása abszolút ÉS relatív mértékben.

        A kiírás kötelezően előírja mindkettőt, különben a módszerek erőforrásigénye
        nem összehasonlítható.
        """
        stored = int(sum(v["x"].shape[0] for v in self.store.values()))
        cumulative = int(sum(self.task_train_sizes.get(t, 0) for t in seen_tasks))
        return {
            "buffer_total_capacity": int(self.total_size),
            "stored_samples": stored,
            "stored_per_task": {t: int(v["x"].shape[0]) for t, v in self.store.items()},
            "cumulative_train_samples": cumulative,
            "relative_ratio_percent": round(100.0 * stored / cumulative, 4) if cumulative else 0.0,
            "memory_bytes_float32": stored * 3 * 28 * 28 * 4,
        }

## 3.2 Regularizációs módszer: Elastic Weight Consolidation (EWC)

A regularizációs eljárások **teljesen adatmentesek**: nem tárolnak nyers képeket a korábbi feladatokból, hanem a veszteségfüggvényt egészítik ki egy olyan büntetőtaggal, amely megakadályozza a korábbi feladatok szempontjából kulcsfontosságú tudás felülírását.

Az EWC a tanítás legvégén készít egy mérést: **fontossági térképet** épít arról, hogy az egyes súlyok mennyire kritikusak az adott feladathoz. A fontosságot a veszteségfüggvény görbülete, azaz a **diagonális Fisher-információ** becsli:

$$F_n = \mathbb{E}\left[ \left( \frac{\partial \log p(y \mid x, \theta)}{\partial \theta_n} \right)^{\!2} \right]$$

Ha a modell egy új feladatot tanul, és egy kritikus súlyt akarna megváltoztatni, a büntetőtag ezt visszatartja; a szabadon felhasználható súlyoknál nincs ilyen védelem:

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\tau_i}(\theta) + \frac{\lambda}{2} \sum_n F_n \left( \theta_n - \theta^{*}_n \right)^2$$

A $\lambda$ paraméterrel szabályozható, mennyire erős a visszatartás:

* **túl alacsony $\lambda$:** a modell továbbra is sokat felejt,
* **túl magas $\lambda$:** a modell megmerevedik, és nem tud új feladatot tanulni.

### Implementációs döntések

* **Csak a közös gerinchálózatot védjük.** A task-specifikus fejek úgysem íródnak felül: a forward hívásban csak az aktuális feladat feje fut le, a többi fej gradiense `None` marad, így az optimizer át is ugorja őket. A felejtés kizárólag a backbone elmászásából ered.
* **Empirikus Fisher.** A valódi címkék log-likelihoodjának gradiensnégyzetéből számoljuk, a tanítóhalmaz első `fisher_batches` batchén. Ez a gyakorlatban elterjedt közelítés, a teljes halmazon való számítás aránytalanul drága lenne.
* **Online akkumuláció.** A korábbi feladatok Fisher-mátrixát nem dobjuk el, hanem összegezzük, a referenciapont ($\theta^{*}$) pedig mindig a legutóbbi feladat utáni súlyállapot.

> **Ismert kockázat (jegyzet 6. oldal):** minden feladat után újabb súlyokat rögzítünk, ezért sok feladat esetén kapacitáskimerülés léphet fel: a modell fokozatosan megmerevedik, és az utolsó feladatokat már nem tudja megtanulni. Ugyanez igaz az SI eljárásra is.

**Megjegyzés:** a jegyzet a Synaptic Intelligence (SI) eljárást is tárgyalja, amely ugyanezt a fontossági elvet folyamatosan, a teljes tanulási út mentén számolja. A kötelező minimum egyetlen regularizációs módszert ír elő, ezért az SI implementálása opcionális továbbfejlesztés maradt.

In [ ]:
class EWC:
    """
    Elastic Weight Consolidation.

    Minden feladat VÉGÉN kiszámoljuk, mely súlyok voltak kritikusak az adott feladathoz
    (diagonális Fisher-információ), és a következő feladatok lossához hozzáadunk egy
    büntetőtagot, amely ezeket a súlyokat a helyükön tartja:

        L_total = L_uj + (lambda / 2) * sum_n  F_n * (theta_n - theta*_n)^2

    Csak a KÖZÖS gerinchálózat paramétereit védjük: a task-specifikus fejek és adapterek
    nem íródnak felül, mert a nem használt modulok gradiense None marad.
    """

    def __init__(self, lam=5000.0, fisher_batches=50):
        self.lam = lam
        self.fisher_batches = fisher_batches
        self.fisher = {}        # paraméternév -> akkumulált Fisher-diagonális
        self.star = {}          # paraméternév -> a legutóbbi feladat utáni súlyérték
        self.n_tasks_seen = 0

    @staticmethod
    def _shared_named_parameters(model):
        """A közös (nem feladatspecifikus) paraméterek bejárása."""
        for name, p in model.named_parameters():
            if name.startswith("heads.") or name.startswith("adapters."):
                continue
            if p.requires_grad:
                yield name, p

    def update(self, model, train_loader, task_name, task_type):
        """Feladat lezárása: Fisher-mátrix becslése és a referenciapont rögzítése."""
        criterion = get_criterion(task_type)
        was_training = model.training
        model.eval()

        fisher_task = {n: torch.zeros_like(p) for n, p in self._shared_named_parameters(model)}
        seen_batches = 0

        for images, labels in train_loader:
            if seen_batches >= self.fisher_batches:
                break
            images = images.to(device)
            labels = prepare_labels(labels, task_type)

            model.zero_grad(set_to_none=True)
            loss = criterion(model(images, task_name=task_name), labels)
            loss.backward()

            for n, p in self._shared_named_parameters(model):
                if p.grad is not None:
                    fisher_task[n] += p.grad.detach() ** 2
            seen_batches += 1

        if seen_batches > 0:
            for n in fisher_task:
                fisher_task[n] /= seen_batches

        # Online akkumuláció: a korábbi Fisher-mátrixokat nem dobjuk el
        for n, f in fisher_task.items():
            self.fisher[n] = self.fisher.get(n, torch.zeros_like(f)) + f

        # A referenciapont mindig a legutóbbi feladat utáni súlyállapot
        self.star = {n: p.detach().clone() for n, p in self._shared_named_parameters(model)}
        self.n_tasks_seen += 1

        model.zero_grad(set_to_none=True)
        if was_training:
            model.train()

    def penalty(self, model):
        """A büntetőtag kiszámítása az aktuális súlyállapotra."""
        if not self.fisher:
            return torch.zeros((), device=device)
        total = torch.zeros((), device=device)
        for n, p in self._shared_named_parameters(model):
            if n in self.fisher and n in self.star:
                total = total + (self.fisher[n] * (p - self.star[n]) ** 2).sum()
        return 0.5 * self.lam * total

    def importance_report(self):
        """Diagnosztika: mekkora a felhalmozott fontossági tömeg."""
        if not self.fisher:
            return {"tasks_consolidated": 0, "fisher_mass": 0.0, "protected_parameters": 0}
        return {
            "tasks_consolidated": self.n_tasks_seen,
            "fisher_mass": float(sum(f.sum().item() for f in self.fisher.values())),
            "protected_parameters": int(sum(f.numel() for f in self.fisher.values())),
        }

## 3.3 Parameter isolation: részleges fagyasztás és adapterek

A parameter-isolation eljárások adatmentesek, és a problémát a gyökerénél fogják meg: ha két feladat fizikailag **nem ugyanazokat a paramétereket írja át**, akkor nem is tudják felülírni egymás tudását. A replayhez képest nem igényelnek memóriát, a regularizációhoz képest pedig nem merevedik le a teljes hálózat sok feladat után.

### A) Partial freezing

A neurális háló bizonyos rétegeit a pontosság javítása érdekében befagyasztjuk: a befagyasztott rétegekben gradienst és súlyfrissítést nem alkalmazunk. A módszer nem jár memória- és paraméternövekedéssel, mivel a fagyasztott súlyoknál csökken a gradiens-visszaterjesztés (backpropagation) láncolata, így csökken a számítási igény és a tanítási idő is.

A módszer teljesítménye **kritikusan függ az első feladat adateloszlásának gazdagságától**, hiszen a befagyasztott reprezentáció minden későbbi feladatot ki kell hogy szolgáljon. Diagnosztika a hangoláshoz:

* ha az **FWT** (átvihetőség) esett vissza túlzottan, túl sok réteget fagyasztottunk be,
* ha a **BWT** (felejtés) alig javult, túl keveset.

### B) Adapterek

Az adapteres megoldás továbbviszi a fagyasztás gondolatát, és kiküszöböli a megosztott súlyok felülírását azzal, hogy a paramétereket két szigorúan elkülönülő halmazra bontja:

1. **Univerzálisan befagyasztott gerinchálózat.** Az első feladat tanítása után a teljes közös gerinc súlyait véglegesen lezárjuk; ez szolgál invariáns, általános vizuális jellemzőkinyerőként.
2. **Könnyűsúlyú, feladatspecifikus adapter modulok.** Minden új feladathoz a gerinc blokkjai közé vagy mellé apró, kisméretű dedikált számítási egységeket illesztünk be. Új feladat érkezésekor nem a gerinc súlyait írjuk át, hanem új modulokat tanítunk be. A leggyakoribb választás egy párhuzamos 1×1-es konvolúciós adapter:

$$y = x + W_{1\times1}^{(\tau)}(x)$$

Ezzel szeparált tanulás érhető el, mivel minden feladat esetén kizárólag a hozzá tartozó adapter modul és kimeneti fej súlyai változnak.

* **Előny:** mivel minden feladatnak saját adaptere van, a korábban megtanult tudás nem tud felülíródni; a felejtés minimálisra csökken.
* **Hátrány:** ahogy egyre több feladatot tanítunk be, a paraméterek száma folyamatosan nő, ezért a paraméternövekedést mérni kell.

### Két implementációs részlet, ami nélkül a módszer nem működik

**1. Zero-init adapter.** Az 1×1 konvolúció súlyait nullára inicializáljuk, így a tanítás kezdetén az adapter pontosan identitásleképezés. A befagyasztott gerinc viselkedése ezzel bitre változatlan marad, és az adapter csak annyit tér el tőle, amennyit a saját feladata megkövetel.

**2. BatchNorm eval módban.** A `requires_grad = False` **nem** állítja meg a BatchNorm rétegek futó statisztikáinak (`running_mean`, `running_var`) frissítését, mert azok nem paraméterek, hanem pufferek. Ha a BN réteg tanító módban marad, a „befagyasztott” gerinc kimenete az új feladat adatain akkor is elmászik, és a nulla felejtés ígérete nem teljesül. Ezért minden `model.train()` hívás után a gerinc BN rétegeit külön `eval()` módba kell állítani.

In [ ]:
class TaskAdapter(nn.Module):
    """
    Párhuzamos 1x1 konvolúciós adapter modul:  y = x + W(x)

    A W konvolúció súlyai NULLÁRA vannak inicializálva, ezért az adapter a tanítás
    kezdetén pontosan identitásleképezés. Így a befagyasztott gerinchálózat viselkedése
    változatlan marad, és az adapter csak annyit tér el tőle, amennyit a saját feladata
    megkövetel.
    """

    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=1, bias=False)
        nn.init.zeros_(self.conv.weight)

    def forward(self, x):
        return x + self.conv(x)


class AdapterResNet18(nn.Module):
    """
    Ugyanaz a ResNet-18 gerinc és ugyanazok a task-specifikus fejek, mint a
    MultiHeadResNet18-ban, de a stage-ek (layer1..layer4) kimenete után beilleszthető
    egy feladatspecifikus adapter modul.

    use_adapters=False esetén ez pontosan a "partial freezing" ág: nincs extra modul,
    csak a befagyasztott gerinc és a feladatspecifikus kimeneti fejek. A két ág így
    ugyanannak a mechanizmusnak a két változata, és közvetlenül összevethető.
    """

    STAGE_CHANNELS = [64, 128, 256, 512]

    def __init__(self, task_classes_dict, use_adapters=True):
        super().__init__()
        base = models.resnet18(weights=None)

        self.stem = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4
        self.avgpool = base.avgpool
        in_features = base.fc.in_features

        self.use_adapters = use_adapters

        self.heads = nn.ModuleDict({
            task_name: nn.Linear(in_features, num_classes)
            for task_name, num_classes in task_classes_dict.items()
        })

        if use_adapters:
            self.adapters = nn.ModuleDict({
                task_name: nn.ModuleList([TaskAdapter(c) for c in self.STAGE_CHANNELS])
                for task_name in task_classes_dict
            })

    def backbone_modules(self):
        """A közös, fagyasztható részek (a fejek és az adapterek NEM tartoznak ide)."""
        return [self.stem, self.layer1, self.layer2, self.layer3, self.layer4]

    def forward(self, x, task_name):
        if task_name not in self.heads:
            raise ValueError(f"Ismeretlen task név: {task_name}")

        adapters = self.adapters[task_name] if self.use_adapters else None

        x = self.stem(x)
        for i, layer in enumerate([self.layer1, self.layer2, self.layer3, self.layer4]):
            x = layer(x)
            if adapters is not None:
                x = adapters[i](x)

        x = torch.flatten(self.avgpool(x), 1)
        return self.heads[task_name](x)

    def backbone_parameter_count(self):
        return int(sum(p.numel() for m in self.backbone_modules() for p in m.parameters()))

    def adapter_parameter_count(self):
        if not self.use_adapters:
            return 0
        return int(sum(p.numel() for p in self.adapters.parameters()))


def freeze_backbone(model):
    """A közös gerinc súlyainak befagyasztása (a fejek és adapterek taníthatók maradnak)."""
    for module in model.backbone_modules():
        for p in module.parameters():
            p.requires_grad = False


def set_backbone_bn_eval(model):
    """
    FONTOS: a requires_grad=False NEM állítja meg a BatchNorm futó statisztikáit,
    mert azok pufferek, nem paraméterek. Ha a BN réteg train módban marad, a
    befagyasztott gerinc kimenete az új feladat adatain akkor is elmászik, és a
    nulla felejtés ígérete nem teljesül. Ezért minden model.train() hívás után
    a gerinc BN rétegeit külön eval() módba kell tenni.
    """
    for module in model.backbone_modules():
        for m in module.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()

## 3.4 Egységes futtatókeret

A `run_continual_experiment` függvény mind az öt continual ágat kiszolgálja. A különbség mindössze három beavatkozási pont:

| Beavatkozás | `sequential` | `replay` | `ewc` | `freeze` | `adapter` |
|---|---|---|---|---|---|
| Modellosztály | `MultiHeadResNet18` | `MultiHeadResNet18` | `MultiHeadResNet18` | `AdapterResNet18(use_adapters=False)` | `AdapterResNet18(use_adapters=True)` |
| Loss kiegészítés | – | memória-loss | Fisher-büntetőtag | – | – |
| Optimizer hatóköre | minden paraméter | minden paraméter | minden paraméter | gerinc nélkül (1. task után) | gerinc nélkül (1. task után) |
| Feladat lezárása után | – | memória feltöltése | Fisher-becslés | – | – |

Minden ág ugyanazt a kiértékelési ciklust futtatja és ugyanazokat a mentési lépéseket végzi el.

### Mentett kimenetek futásonként

A reprodukálhatósági minimum szerint az eredményeket nyers formában is menteni kell, nem csak ábrákban, és a táblázatoknak újragenerálhatónak kell lenniük a mentett fájlokból:

| Fájl | Tartalom |
|---|---|
| `wp3_<módszer>_seed42_R_matrix.csv` | a nyers $T \times T$ mátrix |
| `wp3_<módszer>_seed42_config.json` | a teljes futási konfiguráció és a futásnapló |
| `wp3_<módszer>_seed42_metrics.json` | $A_T$, BWT, felejtés, FWT (a WP1 metrikafüggvényével) |
| `wp3_<módszer>_seed42_heatmap.png` | a mátrix hőtérképe (a WP1 ábrafüggvényével) |

A futásnapló feladatonként rögzíti a tanítási időt, a paraméterszámot, a replay memóriahasználatát (abszolút és relatív) és az adapterek paraméternövekedését. Ezekre a 4. munkacsomag erőforrás-elemzése épül majd.

In [ ]:
def run_continual_experiment(method,
                             task_order=None,
                             seed=42,
                             cfg=None,
                             b_single_dict=None,
                             experiment_name=None,
                             verbose=True):
    """
    Egységes futtatókeret a 3. munkacsomag continual-learning módszereihez.

    method:
      "sequential" - baseline, semmilyen védelem
      "replay"     - fix méretű, class-balanced exemplar memória
      "ewc"        - Elastic Weight Consolidation (regularizációs)
      "freeze"     - parameter isolation: a gerinc befagyasztása az 1. feladat után
      "adapter"    - parameter isolation: befagyasztott gerinc + feladatspecifikus adapterek

    Minden ág ugyanazt a T x T mátrixot tölti ki, ugyanazzal a kiértékelési protokollal,
    és ugyanazokat a metrikákat menti, így a módszerek közvetlenül összehasonlíthatók.
    """
    cfg = cfg if cfg is not None else WP3_CONFIG
    task_order = task_order if task_order is not None else cfg["task_order"]
    task_classes = cfg["task_classes"]
    experiment_name = experiment_name or f"wp3_{method}"
    batch_size = cfg["batch_size"]
    eval_drop_last = cfg["eval_drop_last"]

    if method not in ("sequential", "replay", "ewc", "freeze", "adapter"):
        raise ValueError(f"Ismeretlen módszer: {method}")

    set_seed(seed)
    T = len(task_order)
    R_matrix = np.zeros((T, T))

    # --- Modell kiválasztása ---------------------------------------------------
    uses_isolation = method in ("freeze", "adapter")
    if uses_isolation:
        model = AdapterResNet18(task_classes, use_adapters=(method == "adapter")).to(device)
    else:
        model = MultiHeadResNet18(task_classes).to(device)

    # --- Módszerspecifikus állapot ---------------------------------------------
    buffer = ExemplarBuffer(cfg["replay"]["buffer_total_size"], seed=seed) if method == "replay" else None
    ewc = EWC(cfg["ewc"]["lambda"], cfg["ewc"]["fisher_batches"]) if method == "ewc" else None

    # Adatok előkészítése és gyorsítótárazása (egyben a letöltés is itt történik meg)
    bundles = {t: get_cl_data(t, batch_size, eval_drop_last) for t in task_order}
    task_types = {t: bundles[t]["info"]["task"] for t in task_order}
    criteria = {t: get_criterion(task_types[t]) for t in task_order}

    run_log = {"method": method, "seed": seed, "per_task": []}
    total_start = time.time()

    # =========================== TANÍTÁSI CIKLUS ================================
    for i, train_task in enumerate(task_order):
        if verbose:
            print(f"\n  Fázis [{i+1}/{T}] - {method} - tanítás: {train_task.upper()}")

        # Parameter isolation: az ELSŐ feladat után lezárjuk a közös gerincet
        if uses_isolation and i == cfg["adapter"]["backbone_frozen_after_task_index"]:
            freeze_backbone(model)
            if verbose:
                print(f"    -> gerinc befagyasztva "
                      f"({count_parameters(model, only_trainable=True):,} tanítható paraméter maradt)")

        trainable = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.Adam(trainable, lr=cfg["learning_rate"])

        task_start = time.time()
        for epoch in range(1, cfg["epochs_per_task"] + 1):
            model.train()
            if uses_isolation and i >= cfg["adapter"]["backbone_frozen_after_task_index"]:
                set_backbone_bn_eval(model)

            for images, labels in bundles[train_task]["train_loader"]:
                images = images.to(device)
                labels = prepare_labels(labels, task_types[train_task])
                optimizer.zero_grad(set_to_none=True)

                loss = criteria[train_task](model(images, task_name=train_task), labels)

                # --- REPLAY: a memóriából vett régi minták lossa hozzáadódik -----
                if buffer is not None and len(buffer.store) > 0:
                    for old_task, old_x, old_y in buffer.sample(batch_size):
                        old_x = old_x.to(device)
                        old_y = prepare_labels(old_y, task_types[old_task])
                        loss = loss + criteria[old_task](model(old_x, task_name=old_task), old_y)

                # --- EWC: büntetőtag a korábbi feladatok kritikus súlyaira -------
                if ewc is not None and i > 0:
                    loss = loss + ewc.penalty(model)

                loss.backward()
                optimizer.step()

            val_metrics = evaluate_detailed(model, bundles[train_task]["val_loader"],
                                            task_types[train_task], criteria[train_task],
                                            eval_task=train_task)
            if verbose:
                print(f"    epoch {epoch}/{cfg['epochs_per_task']} - "
                      f"Val Acc ({train_task}): {val_metrics['accuracy']:.2f}%")

        task_seconds = time.time() - task_start

        # --- Feladat lezárása: memória / Fisher frissítése -----------------------
        if buffer is not None:
            buffer.add_task(train_task, bundles[train_task]["train_dataset"])
        if ewc is not None:
            ewc.update(model, bundles[train_task]["train_loader"], train_task, task_types[train_task])

        # --- Kiértékelés az ÖSSZES feladaton (a mátrix i-edik sora) --------------
        model.eval()
        with torch.no_grad():
            for j, eval_task in enumerate(task_order):
                m = evaluate_detailed(model, bundles[eval_task]["test_loader"],
                                      task_types[eval_task], criteria[eval_task],
                                      eval_task=eval_task)
                R_matrix[i, j] = m["accuracy"]

        # --- Erőforrás-napló (a 4. munkacsomag ezekre épül) ----------------------
        entry = {
            "step": i + 1,
            "task": train_task,
            "train_seconds": round(task_seconds, 2),
            "trainable_parameters": count_parameters(model, only_trainable=True),
            "total_parameters": count_parameters(model),
        }
        if buffer is not None:
            entry["memory"] = buffer.memory_report(task_order[:i + 1])
        if ewc is not None:
            entry["ewc"] = ewc.importance_report()
        if uses_isolation:
            backbone_params = model.backbone_parameter_count()
            adapter_params = model.adapter_parameter_count()
            entry["backbone_parameters"] = backbone_params
            entry["adapter_parameters_total"] = adapter_params
            entry["adapter_parameters_per_task"] = adapter_params // max(1, T)
            entry["parameter_growth_percent"] = (
                round(100.0 * adapter_params / backbone_params, 4) if backbone_params else 0.0
            )
        run_log["per_task"].append(entry)

    run_log["total_seconds"] = round(time.time() - total_start, 2)

    # =========================== MENTÉS =========================================
    # 1. Nyers T x T mátrix CSV-ben (a kiírás megköveteli a nyers eredményt)
    df_R = pd.DataFrame(R_matrix,
                        index=[f"after_{t}" for t in task_order],
                        columns=list(task_order))
    raw_path = os.path.join(RESULT_DIR, f"{experiment_name}_seed{seed}_R_matrix.csv")
    df_R.to_csv(raw_path)

    # 2. Teljes futási konfiguráció és napló (reprodukálhatósági minimum)
    cfg_path = os.path.join(RESULT_DIR, f"{experiment_name}_seed{seed}_config.json")
    with open(cfg_path, "w", encoding="utf-8") as f:
        json.dump({"method": method, "seed": seed, "task_order": list(task_order),
                   "config": cfg, "run_log": run_log},
                  f, indent=4, ensure_ascii=False, default=str)

    # 3. Hőtérkép és metrikák a WP1-ben megírt függvényekkel
    save_publication_matrix(R_matrix=R_matrix, task_order=list(task_order),
                            experiment_name=f"{experiment_name}_seed{seed}")
    metrics = compute_and_save_cl_metrics(R_matrix=R_matrix, task_order=list(task_order),
                                          seed=seed, experiment_name=experiment_name,
                                          b_single_dict=b_single_dict)

    print(f"  -> nyers mátrix: {raw_path}")
    print(f"  -> konfiguráció: {cfg_path}")
    print(f"  -> A_T = {metrics['final_average_accuracy']:.2f}% | "
          f"BWT = {metrics['backward_transfer']:.2f}% | "
          f"Felejtés = {metrics['average_forgetting']:.2f}% | "
          f"idő = {run_log['total_seconds']:.0f}s")

    return {"metrics": metrics, "R_matrix": R_matrix, "run_log": run_log, "model": model}

## 3.5 Joint training: felső referencia

A joint training azt vizsgálja, hogy mire képes a neurális háló, ha egyáltalán nincs időbeli korlát, és az összes feladat adatát egyszerre látja. Mivel nincs mit elfelejtenie, az itt elért pontosság a hálózat elméleti maximuma a kapott architektúra mellett.

Ezzel megmérhető a **tudásvesztési rés**:

$$\Delta_{\text{gap}} = A_T^{\text{joint}} - A_T^{\text{CL}}$$

* $A_T^{\text{joint}}$: a felső referencia értéke,
* $A_T^{\text{CL}}$: a continual learning során elért pontosság.

Azt mérjük, hogy a felejtés miatt mennyi pontosságot veszítünk, azaz milyen közel tud kerülni a hálózat az elméleti maximumához.

> **Figyelem:** a joint training a feladatkiírás szerint sem tekinthető fair continual baseline-nak, mert az összes adatot egyszerre látja. Kizárólag felső referenciaként szabad hivatkozni rá.

### A közös adathalmaz létrehozásának problémája

Nem omleszthetjük be a feladatokat egyetlen nagy adathalmazba: akkor egy epoch alatt a nagy mennyiségű adathalmaz elemeit többször látja a modell, mint a kisebb elemszámú adathalmaz elemeit. A MedMNIST feladatok mérete 4 708 (PneumoniaMNIST) és 97 477 (OCTMNIST) között szór, tehát egyszerű konkatenálásnál a nagy halmazok elnyomnák a kicsiket.

**Megoldás: round-robin batching.** Minden tanítási lépésben minden adathalmazból külön-külön veszünk egy-egy azonos méretű mini-batchet, kiszámoljuk mindazok lossát, majd azokat összeadjuk, és egyszerre lépünk a gradienssel:

$$\mathcal{L}_{\text{joint}} = \sum_{t=1}^{T} \mathcal{L}_{\tau_t}\big(x^{(t)}, y^{(t)}\big)$$

A kisebb adathalmazok loaderei körbefordulnak, amíg a leghosszabb végig nem ér; egy epoch hossza a leghosszabb loader hossza.

In [ ]:
def run_joint_training(task_order=None, seed=42, cfg=None, verbose=True):
    """
    Joint training - NEM continual baseline, hanem FELSŐ REFERENCIA.

    A modell egyszerre látja az összes feladat adatát, tehát nincs mit elfelejtenie.
    Az itt kapott átlagos pontosság és a continual A_T különbsége a tudásvesztési rés:

        delta_gap = A_T_joint - A_T_CL

    Round-robin batching: minden lépésben MINDEN adathalmazból veszünk egy azonos
    méretű mini-batchet, a lossokat összeadjuk, és egyszerre lépünk a gradienssel.
    Így a nagy adathalmazok (OCTMNIST ~97k) nem nyomják el a kicsiket
    (PneumoniaMNIST ~4.7k), ami egyszerű konkatenálásnál megtörténne.
    """
    cfg = cfg if cfg is not None else WP3_CONFIG
    task_order = task_order if task_order is not None else cfg["task_order"]
    batch_size = cfg["batch_size"]
    eval_drop_last = cfg["eval_drop_last"]

    set_seed(seed)
    model = MultiHeadResNet18(cfg["task_classes"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["learning_rate"])

    bundles = {t: get_cl_data(t, batch_size, eval_drop_last) for t in task_order}
    task_types = {t: bundles[t]["info"]["task"] for t in task_order}
    criteria = {t: get_criterion(task_types[t]) for t in task_order}

    steps_per_epoch = max(len(bundles[t]["train_loader"]) for t in task_order)
    start = time.time()

    for epoch in range(1, cfg["epochs_per_task"] + 1):
        model.train()
        iterators = {t: iter(bundles[t]["train_loader"]) for t in task_order}

        for step in range(steps_per_epoch):
            optimizer.zero_grad(set_to_none=True)
            total_loss = torch.zeros((), device=device)

            for t in task_order:
                try:
                    images, labels = next(iterators[t])
                except StopIteration:
                    # A kisebb adathalmazok körbefordulnak (round-robin)
                    iterators[t] = iter(bundles[t]["train_loader"])
                    images, labels = next(iterators[t])

                images = images.to(device)
                labels = prepare_labels(labels, task_types[t])
                total_loss = total_loss + criteria[t](model(images, task_name=t), labels)

            total_loss.backward()
            optimizer.step()

        if verbose:
            print(f"  [joint] epoch {epoch}/{cfg['epochs_per_task']} kész "
                  f"({steps_per_epoch} lépés x {len(task_order)} feladat)")

    # --- Kiértékelés az összes feladat teszthalmazán ---------------------------
    model.eval()
    per_task = {}
    with torch.no_grad():
        for t in task_order:
            m = evaluate_detailed(model, bundles[t]["test_loader"], task_types[t],
                                  criteria[t], eval_task=t)
            per_task[t] = float(m["accuracy"])

    joint_summary = {
        "experiment_name": "wp3_joint",
        "seed": seed,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "note": "FELSŐ REFERENCIA, nem fair continual baseline: minden adatot egyszerre lát.",
        "strategy": "round-robin batching",
        "steps_per_epoch": int(steps_per_epoch),
        "epochs": cfg["epochs_per_task"],
        "task_order": list(task_order),
        "per_task_accuracy": {k: round(v, 4) for k, v in per_task.items()},
        "joint_average_accuracy": round(float(np.mean(list(per_task.values()))), 4),
        "total_parameters": count_parameters(model),
        "train_seconds": round(time.time() - start, 2),
    }

    path = os.path.join(RESULT_DIR, f"wp3_joint_seed{seed}_metrics.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(joint_summary, f, indent=4, ensure_ascii=False)

    print(f"  -> joint felső referencia: A_T_joint = "
          f"{joint_summary['joint_average_accuracy']:.2f}%")
    print(f"  -> mentve: {path}")

    return joint_summary

## 3.6 A kísérletek futtatása

A 2. munkacsomag konvenciójával azonosan az **eredeti task sorrenden**, **seed 42** mellett futtatunk. A `single_task_reference.json` az FWT számításához szükséges, ezt a korábbi cella már elmentette.

> **Futásidő:** az `epochs_per_task = 1` beállítás tesztfutásra való. Az `ewc` ág a Fisher-becslés miatt, a `replay` ág a memóriából vett extra forward-backward lépések miatt lassabb a baseline-nál; a `freeze` és `adapter` ágak viszont gyorsabbak, mert a befagyasztott gerincen nem kell gradienst visszaterjeszteni. Éles méréshez az `epochs_per_task` értékét 10-20-ra kell emelni a `WP3_CONFIG`-ban.

In [ ]:
# A single-task referencia betöltése az FWT számításához
_single_ref_path = os.path.join(RESULT_DIR, "single_task_reference.json")
with open(_single_ref_path, "r", encoding="utf-8") as f:
    single_task_reference = json.load(f)
print(f"Single-task referencia betöltve: {single_task_reference}")

WP3_METHODS = ["sequential", "replay", "ewc", "freeze", "adapter"]
wp3_runs = {}

for _method in WP3_METHODS:
    print("\n" + "=" * 70)
    print(f" 3. MUNKACSOMAG - {_method.upper()}  (seed {WP3_CONFIG['seed']})")
    print("=" * 70)
    wp3_runs[_method] = run_continual_experiment(
        method=_method,
        task_order=WP3_TASK_ORDER,
        seed=WP3_CONFIG["seed"],
        cfg=WP3_CONFIG,
        b_single_dict=single_task_reference,
    )

print("\n" + "=" * 70)
print(f" 3. MUNKACSOMAG - JOINT TRAINING (felső referencia)")
print("=" * 70)
wp3_joint = run_joint_training(task_order=WP3_TASK_ORDER,
                               seed=WP3_CONFIG["seed"],
                               cfg=WP3_CONFIG)

## 3.7 Módszerek összehasonlítása

Az összefoglaló táblázat a mentett JSON fájlokból épül újra, tehát a kísérletek újrafuttatása nélkül is regenerálható. Az oszlopok:

* **$A_T$** – végső átlagos pontosság (minél nagyobb, annál jobb),
* **BWT** – visszamenőleges transzfer (negatív érték jelzi a felejtést),
* **$F$** – átlagos felejtés (minél kisebb, annál jobb),
* **FWT (single-ref)** – az izolált modellhez mért előrefelé transzfer,
* **$\Delta_{\text{gap}}$** – a joint felső referenciától mért távolság,
* **erőforrás-oszlopok** – tárolt minták, relatív memóriaarány, paraméterszám és tanítási idő.

Ezek az erőforrás-oszlopok szolgálnak majd a 4. munkacsomag accuracy-vs-resource elemzésének bemenetéül.

In [ ]:
def build_wp3_comparison(methods=None, seed=42, joint_summary=None):
    """
    A 3. munkacsomag összehasonlító táblázata a MENTETT fájlokból építve.

    A kísérletek újrafuttatása nélkül is lefuttatható, mert kizárólag a
    results/ mappába mentett JSON riportokból dolgozik.
    """
    methods = methods or WP3_METHODS

    if joint_summary is None:
        joint_path = os.path.join(RESULT_DIR, f"wp3_joint_seed{seed}_metrics.json")
        if os.path.exists(joint_path):
            with open(joint_path, encoding="utf-8") as f:
                joint_summary = json.load(f)
    joint_acc = joint_summary["joint_average_accuracy"] if joint_summary else None

    rows = []
    for method in methods:
        metrics_path = os.path.join(RESULT_DIR, f"wp3_{method}_seed{seed}_metrics.json")
        config_path = os.path.join(RESULT_DIR, f"wp3_{method}_seed{seed}_config.json")
        if not (os.path.exists(metrics_path) and os.path.exists(config_path)):
            print(f"Hiányzó eredményfájl, kihagyva: {method}")
            continue

        with open(metrics_path, encoding="utf-8") as f:
            m = json.load(f)
        with open(config_path, encoding="utf-8") as f:
            log = json.load(f)["run_log"]

        last = log["per_task"][-1]
        mem = last.get("memory", {})

        rows.append({
            "Módszer": method,
            "A_T (%)": m["final_average_accuracy"],
            "BWT (%)": m["backward_transfer"],
            "Felejtés F (%)": m["average_forgetting"],
            "FWT single (%)": m["forward_transfer_single"],
            "Delta gap (%)": (round(joint_acc - m["final_average_accuracy"], 4)
                              if joint_acc is not None else None),
            "Tárolt minták": mem.get("stored_samples", 0),
            "Memória arány (%)": mem.get("relative_ratio_percent", 0.0),
            "Extra paraméter (%)": last.get("parameter_growth_percent", 0.0),
            "Összes paraméter": last["total_parameters"],
            "Tanítási idő (s)": log["total_seconds"],
        })

    df = pd.DataFrame(rows).sort_values("A_T (%)", ascending=False).reset_index(drop=True)
    csv_path = os.path.join(RESULT_DIR, f"wp3_method_comparison_seed{seed}.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8")
    print(f"Összehasonlító táblázat elmentve: {csv_path}")
    if joint_acc is not None:
        print(f"Joint felső referencia (A_T_joint): {joint_acc:.2f}%")
    return df


def plot_wp3_comparison(df, seed=42):
    """Publikációs minőségű oszlopdiagram: pontosság és felejtés módszerenként."""
    plt.close("all")
    plt.rcParams.update(plt.rcParamsDefault)
    sns.set_theme(style="whitegrid")

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.0), dpi=300)
    fig.patch.set_facecolor("white")

    sns.barplot(data=df, x="Módszer", y="A_T (%)", ax=axes[0],
                hue="Módszer", palette="Blues_d", legend=False)
    axes[0].set_title("Végső átlagos pontosság (A_T)", fontsize=12, fontweight="bold")
    axes[0].set_ylabel("Test Accuracy (%)")

    sns.barplot(data=df, x="Módszer", y="Felejtés F (%)", ax=axes[1],
                hue="Módszer", palette="Reds_d", legend=False)
    axes[1].set_title("Átlagos felejtés (F) - kisebb a jobb", fontsize=12, fontweight="bold")
    axes[1].set_ylabel("Forgetting (%)")

    for ax in axes:
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=20)
        for container in ax.containers:
            ax.bar_label(container, fmt="%.1f", fontsize=9, padding=2)

    plt.tight_layout()
    img_path = os.path.join(RESULT_DIR, f"wp3_method_comparison_seed{seed}.png")
    fig.savefig(img_path, dpi=300, facecolor="white", bbox_inches="tight")
    plt.show()
    print(f"Ábra elmentve: {img_path}")


df_wp3 = build_wp3_comparison(seed=WP3_CONFIG["seed"])
display(df_wp3)
plot_wp3_comparison(df_wp3, seed=WP3_CONFIG["seed"])